# 02 — Evaluación G1: Proyecto Oráculo

**Integrante:** Johan Stiven Osorio Grisales  
**Grupo:** G1 · **Semana:** 2

## Propósito

Cargar la configuración directamente desde `entrega.json`, verificar su estructura y ejecutar la llamada documentada a `oraculo.validar`.

La partición pública disponible contiene 87 instancias de validación. Esta ejecución no incluye ni sustituye la evaluación privada que realizará el docente con sus datos reservados. El equipo no incorpora esos datos al repositorio.

## 1. Preparar repositorio y dependencias

In [ ]:
import os, sys, subprocess, json
from pathlib import Path

REPOSITORIO = "https://github.com/StivenLH24/ADA-2026-2-GP01.git"
RUTA_PROYECTO = Path("/content/ADA-2026-2-GP01")
RAMA = "trabajo-g1"

if not (RUTA_PROYECTO / ".git").exists():
    subprocess.run([
        "git", "clone", "--branch", RAMA, "--single-branch",
        REPOSITORIO, str(RUTA_PROYECTO)
    ], check=True)
else:
    subprocess.run(
        ["git", "pull", "--ff-only", "origin", RAMA],
        cwd=RUTA_PROYECTO, check=True
    )

os.chdir(RUTA_PROYECTO)
sys.path.insert(0, str(RUTA_PROYECTO))
print("Repositorio preparado:", os.getcwd())

In [ ]:
%pip install -q -U "bitsandbytes>=0.46.1" transformers accelerate nltk spacy emoji langdetect immutabledict
!python -m spacy download en_core_web_sm -q

RUTA_VERIFICADORES = "/content/open-instruct"
if not os.path.isdir(os.path.join(RUTA_VERIFICADORES, ".git")):
    !git clone -q https://github.com/allenai/open-instruct /content/open-instruct

if RUTA_VERIFICADORES not in sys.path:
    sys.path.insert(0, RUTA_VERIFICADORES)

import io, urllib.request, zipfile
NLTK_DATA = "https://raw.githubusercontent.com/nltk/nltk_data/gh-pages/packages"
for carpeta, nombre in [
    ("tokenizers", "punkt"),
    ("tokenizers", "punkt_tab"),
    ("taggers", "averaged_perceptron_tagger"),
    ("taggers", "averaged_perceptron_tagger_eng")
]:
    destino = f"/root/nltk_data/{carpeta}"
    os.makedirs(destino, exist_ok=True)
    with urllib.request.urlopen(f"{NLTK_DATA}/{carpeta}/{nombre}.zip") as respuesta:
        contenido = respuesta.read()
    with zipfile.ZipFile(io.BytesIO(contenido)) as archivo:
        archivo.extractall(destino)

print("Dependencias y verificadores preparados.")

## 2. Montar datos y preparar el oráculo

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

CARPETA_DATOS = Path("/content/drive/MyDrive/ADA_G1")
CARPETA_DATOS.mkdir(parents=True, exist_ok=True)
CACHE_EVALUACION = str(CARPETA_DATOS / "cache_evaluacion_G1.json")

import torch
from open_instruct.IFEvalG import instructions_registry
from ayudas import cargar_datos, dividir, cargar_modelo
from oraculo import Oraculo, RANURAS

assert torch.cuda.is_available(), "Seleccionar GPU T4 en Colab."

datos = cargar_datos()
busqueda, validacion = dividir(datos)
print("Instancias de validación visibles:", len(validacion))

modelo = cargar_modelo("qwen17b")
oraculo = Oraculo(
    modelo=modelo,
    datos=busqueda,
    validacion=validacion,
    cache=CACHE_EVALUACION,
    max_new_tokens=384,
    lote=1
)

## 3. Cargar y validar entrega.json
El notebook no redefine la configuración: la lee directamente del archivo del repositorio.

In [ ]:
with open("entrega.json", encoding="utf-8") as f:
    entrega = json.load(f)

assert entrega.get("grupo") == "G1"
assert entrega.get("semana") == 2

config = entrega.get("config")
assert isinstance(config, dict)
assert set(config) == set(RANURAS) | {"temperatura"}
assert all(
    isinstance(config[r], int) and not isinstance(config[r], bool)
    and 0 <= config[r] <= 7
    for r in RANURAS
)
assert config["temperatura"] == 0.0

print("Archivo de entrega válido:")
print(json.dumps(entrega, indent=2, ensure_ascii=False))

## 4. Evaluar la configuración

Se ejecuta `oraculo.validar` sobre la partición configurada en el entorno. En la ejecución de referencia con la partición pública de 87 instancias, el resultado esperado registrado fue 8/87 = 9,2 %.

La evaluación privada del docente no está disponible para los estudiantes y debe realizarse en el entorno de evaluación oficial. Este notebook no inventa ni simula datos privados.

In [ ]:
resultado = oraculo.validar(config, semilla=2026)

correctas = resultado.n - len(resultado.trazas)
fallidas = len(resultado.trazas)

resumen = {
    "grupo": entrega["grupo"],
    "semana": entrega["semana"],
    "configuracion": config,
    "n_evaluadas": resultado.n,
    "correctas": correctas,
    "fallidas": fallidas,
    "precision": resultado.precision,
    "precision_porcentaje": round(100 * resultado.precision, 4)
}

print(json.dumps(resumen, indent=2, ensure_ascii=False))
oraculo._guardar()

## 5. Interpretación
La precisión representa el porcentaje de instancias donde se cumplieron simultáneamente todas las restricciones. Las trazas muestran la primera familia que falló por instancia. La validación visible sirve para comprobar generalización sobre instancias no utilizadas en la búsqueda; no garantiza el resultado en el conjunto privado del docente.

In [ ]:
from collections import Counter
import pandas as pd

frecuencias = Counter(
    traza.get("violo") for traza in resultado.trazas if traza.get("violo")
)
tabla = pd.DataFrame(
    frecuencias.most_common(),
    columns=["familia_primera_falla", "frecuencia"]
)
display(tabla)